## Pipeline Analysis for Lung Cancer Data

This notebook demonstrates MongoDB aggregation pipelines to analyze lung cancer factors, including peer pressure, smoking, and chronic disease.

## Step 1: Import Libraries and Setup MongoDB Connection

Using `utils.py` for MongoDB credentials.

In [1]:
import json
from pymongo import MongoClient
import sys
sys.path.append('..')  
import scripts.utils 

# Connect to MongoDB using the function from utils.py
client = scripts.utils.connect_to_mongo()


INFO:root:Connected to MongoDB successfully.
INFO:root:Connected to MongoDB successfully.


 ## Step 2: Data Insertion

In [2]:

dataset_path = scripts.utils.DATA_PATH  
data = scripts.utils.load_json_data(dataset_path)

# Insert data into MongoDB
db = client['Lungcancer']
patients = db['Patients']

INFO:root:Loaded data from /Users/shaharyaarkutchi/Documents/GitHub/adsc3910-project-group-6/notebooks/../data/dataset.json.


## Step 3: Aggregation Pipelines for Analysis

### Analyze the influence of peer pressure, smoking, and chronic disease on lung cancer occurrences.

In [3]:
# Pipeline to check influence of peer pressure on smoking grouped by gender

def peer_pressure():
    # pipeline
    pipeline = [
        # filtering for peer pressure levels 1 = low and 2 = high
        {
            "$match": { "PEER_PRESSURE": { "$in": [1, 2] } }
        },
        # grouping by gender and peer pressure level to count total smokers
        {
            "$group": {
                "_id": {
                    "gender": "$GENDER",
                    "peer_pressure": "$PEER_PRESSURE"
                },
                "total": { "$sum": 1 },
                "smokers": { "$sum": { "$cond": [ { "$eq": ["$SMOKING", 1] }, 1, 0 ] } }
            }
        },
        # projecting certain fields and smoking percentage
        {
            "$project": {
                "gender": "$_id.gender",
                "peer_pressure": "$_id.peer_pressure",
                "total": 1,
                "smokers": 1,
                "smoking_percentage": { 
                    "$multiply": [ { "$divide": ["$smokers", "$total"] }, 100 ] 
                }
            }
        },
        # sorting by gender and peer pressure level 
        {
            "$sort": { "gender": 1, "peer_pressure": 1 }
        }
    ]

    # aggregating pipeline
    results = list(patients.aggregate(pipeline))

    # results
    for result in results:
        print(f"Gender: {result['gender']}, Peer Pressure Level: {result['peer_pressure']}")
        print(f"  Total Individuals: {result['total']}")
        print(f"  Smokers: {result['smokers']}")
        print(f"  Smoking Percentage: {result['smoking_percentage']:.2f}%")
        print("-" * 40)

# Run peer pressure analysis
peer_pressure()


Gender: F, Peer Pressure Level: 1
  Total Individuals: 3008
  Smokers: 1432
  Smoking Percentage: 47.61%
----------------------------------------
Gender: F, Peer Pressure Level: 2
  Total Individuals: 2936
  Smokers: 1508
  Smoking Percentage: 51.36%
----------------------------------------
Gender: M, Peer Pressure Level: 1
  Total Individuals: 3004
  Smokers: 1532
  Smoking Percentage: 51.00%
----------------------------------------
Gender: M, Peer Pressure Level: 2
  Total Individuals: 3052
  Smokers: 1636
  Smoking Percentage: 53.60%
----------------------------------------


In [4]:
# Lung cancer in smokers by gender
def lc_gender():
    pipeline = [
        { '$group': { '_id': { 'gender': '$GENDER', 'smoking': '$SMOKING' },
                      'total': { '$sum': 1 },
                      'lung_cancer_yes': { '$sum': { '$cond': [ { '$eq': ['$LUNG_CANCER', 'YES'] }, 1, 0 ] } } } },
        { '$project': { 'gender': '$_id.gender', 'smoking': '$_id.smoking', 'total': 1,
                        'lung_cancer_yes': 1,
                        'lung_cancer_percentage': { '$multiply': [ { '$divide': ['$lung_cancer_yes', '$total'] }, 100 ] } } },
        { '$sort': { 'gender': 1, 'smoking': 1 } }
    ]

    results = list(patients.aggregate(pipeline))

    for result in results:
        smoking_status = 'Smoker' if result['smoking'] == 1 else 'Non-Smoker'
        print(f"Gender: {result['gender']}, Smoking Status: {smoking_status}")
        print(f"  Total: {result['total']}, Lung Cancer (YES): {result['lung_cancer_yes']}")
        print(f"  Lung Cancer Percentage: {result['lung_cancer_percentage']:.2f}%")
        print('-' * 40)

# Run lung cancer by gender analysis
lc_gender()


Gender: F, Smoking Status: Non-Smoker
  Total: 751, Lung Cancer (YES): 389
  Lung Cancer Percentage: 51.80%
----------------------------------------
Gender: F, Smoking Status: Smoker
  Total: 2940, Lung Cancer (YES): 1500
  Lung Cancer Percentage: 51.02%
----------------------------------------
Gender: F, Smoking Status: Non-Smoker
  Total: 2253, Lung Cancer (YES): 1167
  Lung Cancer Percentage: 51.80%
----------------------------------------
Gender: M, Smoking Status: Non-Smoker
  Total: 722, Lung Cancer (YES): 367
  Lung Cancer Percentage: 50.83%
----------------------------------------
Gender: M, Smoking Status: Smoker
  Total: 3168, Lung Cancer (YES): 1548
  Lung Cancer Percentage: 48.86%
----------------------------------------
Gender: M, Smoking Status: Non-Smoker
  Total: 2166, Lung Cancer (YES): 1101
  Lung Cancer Percentage: 50.83%
----------------------------------------


In [5]:
# Pipeline to find lung cancer cases in smokers with chronic disease
def lc_smokers_chronic_disease():    
    pipeline = [
        { '$match': { 'SMOKING': 1, 'CHRONIC_DISEASE': { '$in': [1, 2] } } },
        { '$group': { '_id': { 'gender': '$GENDER', 'lung_cancer': '$LUNG_CANCER' },
                      'total': { '$sum': 1 } } },
        { '$group': { '_id': '$_id.gender', 'total': { '$sum': '$total' },
                      'lung_cancer_yes': { '$sum': { '$cond': [{ '$eq': ['$_id.lung_cancer', 'YES'] }, '$total', 0] } } } },
        { '$project': { 'gender': '$_id', 'total': 1, 'lung_cancer_yes': 1,
                        'lung_cancer_percentage': { '$multiply': [ { '$divide': ['$lung_cancer_yes', '$total'] }, 100 ] } } },
        { '$sort': { 'gender': 1 } }
    ]

    # Run pipeline and display results
    results = list(patients.aggregate(pipeline))

    for result in results:
        print(f"Gender: {result['gender']}")
        print(f"  Total Smokers with Chronic Disease: {result['total']}")
        print(f"  Lung Cancer (YES): {result['lung_cancer_yes']}")
        print(f"  Lung Cancer Percentage: {result['lung_cancer_percentage']:.2f}%")
        print('-' * 40)

lc_smokers_chronic_disease()

Gender: F
  Total Smokers with Chronic Disease: 2940
  Lung Cancer (YES): 1500
  Lung Cancer Percentage: 51.02%
----------------------------------------
Gender: M
  Total Smokers with Chronic Disease: 3168
  Lung Cancer (YES): 1548
  Lung Cancer Percentage: 48.86%
----------------------------------------
